In [5]:
import re
import ipaddress
from urllib.parse import urlparse

def is_phishing_url(url):
    indicators = []
    parsed_url = urlparse(url)
    domain = parsed_url.netloc
    path = parsed_url.path

    # Indicator 1: IP address in hostname
    try:
        ipaddress.ip_address(domain)
        indicators.append("IP address in hostname")
    except ValueError:
        pass

    # Indicator 2: Very long URL
    if len(url) > 70:
        indicators.append("Excessively long URL")

    # Indicator 3: '@' symbol in URL
    if '@' in domain:
        indicators.append("\'@\' symbol in domain")

    # Indicator 4: Multiple subdomains or hyphens (simple check)
    if domain.count('.') > 3 or '-' in domain:
        indicators.append("Multiple subdomains or hyphens")

    # Indicator 5: Suspicious keywords
    suspicious_keywords = ['login', 'bank', 'account', 'verify', 'update', 'secure', 'paypal', 'ebay', 'amazon', 'admin']
    for keyword in suspicious_keywords:
        if keyword in domain.lower() or keyword in path.lower():
            indicators.append(f"Suspicious keyword: '{keyword}'")
            break

    # Indicator 6: Redirect using // in path
    if '//' in path.replace('://', ''):
        indicators.append("Double slash (//) in path")

    if indicators:
        print(f"Suspicious URL detected: {url}")
        for indicator in indicators:
            print(f"  - Indicator: {indicator}")
        return True
    else:
        return False

# --- Demonstration ---
test_urls = [
    "https://www.google.com/search?q=legitimate+website",
    "http://192.168.1.1/login.php",
    "https://secure.update.my-bank.com.phishing.net/account/verify.html",
    "http://phishing.site//malicious/login.php",
    "https://example.com/login?username=user&password=password&redirect_to=malicious.site",
    "https://www.legit-site.com/products/view?id=123",
    "http://www.paypal.com@malicious.com/login",
    "https://secure-login.bankofamerica.com/verify-account/",
    "https://www.trustworthy-domain.com/path/to/page"
]

for url in test_urls:
    print(f"\nChecking URL: {url}")
    if is_phishing_url(url):
        print("\tResult: LIKELY PHISHING")
    else:
        print("\tResult: Looks SAFE")


Checking URL: https://www.google.com/search?q=legitimate+website
	Result: Looks SAFE

Checking URL: http://192.168.1.1/login.php
Suspicious URL detected: http://192.168.1.1/login.php
  - Indicator: IP address in hostname
  - Indicator: Suspicious keyword: 'login'
	Result: LIKELY PHISHING

Checking URL: https://secure.update.my-bank.com.phishing.net/account/verify.html
Suspicious URL detected: https://secure.update.my-bank.com.phishing.net/account/verify.html
  - Indicator: Multiple subdomains or hyphens
  - Indicator: Suspicious keyword: 'bank'
	Result: LIKELY PHISHING

Checking URL: http://phishing.site//malicious/login.php
Suspicious URL detected: http://phishing.site//malicious/login.php
  - Indicator: Suspicious keyword: 'login'
  - Indicator: Double slash (//) in path
	Result: LIKELY PHISHING

Checking URL: https://example.com/login?username=user&password=password&redirect_to=malicious.site
Suspicious URL detected: https://example.com/login?username=user&password=password&redirec